# Multiscale Organization in Sequence Representations - Learning Encodes Remote Homology.

Sequences from the SCOPe dataset, clustered into family, superfamily, and fold levels, are used to evaluate whether the model learns structural information . To test this, each sequence is passed through the Transformer, and its final hidden states are mean-pooled to generate a single high-dimensional embedding vector. The core hypothesis is that structurally similar proteins will lie closer together in this embedding space.

The evaluation operates as an unsupervised nearest-neighbor search: every domain in the dataset acts as a query, and all other domains are ranked by their vector distance to it . The authors evaluated both L2 distance (which optimized AUC) and cosine similarity (which optimized Hit-10), with the latter generally used for ranking in the reported results. To ensure the test measures remote homology rather than trivial sequence identity, strict exclusions are applied during ranking. For example, when testing at the superfamily level, candidates from the same family are filtered out, leaving only distant relatives as valid targets. Candidates sharing the query's superfamily are labeled as positive matches, and the model's performance is quantified by averaging AUC and Hit-10 scores across all queries. This is done for all Evalutation/Benchamrk models and as a baseline HHBlits is used

In [1]:
# Needed to import modules from helpers
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import esm
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import train_test_split

current_dir = os.getcwd()
# Gehe einen Ordner nach oben
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from helpers import helper

# Reproducibility

In [2]:
SEED = 42
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(SEED)
print(f"Seed set to {SEED}. Ready to run.")

Seed set to 42. Ready to run.


# Load ESM model to GPU

In [3]:
# kleineres Modell 'esm1_t6_43M_UR50S'
# mittleres Modell "esm1_t12_85M_UR50S"
# größeres Modell "esm1_t33_670M_UR50S"
# esm1_t34_670M_UR50D
# esm1_t34_670M_UR100
# esm1_t34_670M_UR100 modell davor noch randomisieren
# esm1 modell esm1b_t33_650M_UR50S

# Load Pretrained ESM Model 
print("Loading Trained Model...")
model, alphabet = esm.pretrained.esm1_t34_670M_UR50S()

if torch.cuda.is_available():
    model = model.cuda()
    print("Model moved to GPU.")

# # Create Untrained Control Model
# print("Creating Untrained Control Model...")
# model = helper.randomize_model(model)
# if torch.cuda.is_available():
#     model = model.cuda()


Loading Trained Model...


# Get Data

In [4]:
# Download, Parse and Filter SCOP data
helper.download_no_requests()
df_scop = helper.parse_and_filter_scop()

if not df_scop.empty:
    print(f"Total sequences/domains retained: {len(df_scop)}")
    print(df_scop.head())

File already exists. Skipping download.
Parsing and filtering sequences...
Total sequences/domains retained: 12261
  domain_id                                           sequence class fold  \
0   d1dlwa_  slfeqlggqaavqavtaqfyaniqadatvatffngidmpnqtnkta...     a  a.1   
1   d2gkma_  gllsrlrkrepisiydkiggheaievvvedffvrvladdqlsaffs...     a  a.1   
2   d1ngka_  ksfydavggaktfdaivsrfyaqvaedevlrrvypeddlagaeerl...     a  a.1   
3   d2bkma_  eqwqtlyeaiggeetvaklveafyrrvaahpdlrpifpddltetah...     a  a.1   
4   d4i0va_  aslyeklggaaavdlavekfygkvladervnrffvntdmakqkqhq...     a  a.1   

  superfamily   family  
0       a.1.1  a.1.1.1  
1       a.1.1  a.1.1.1  
2       a.1.1  a.1.1.1  
3       a.1.1  a.1.1.1  
4       a.1.1  a.1.1.1  


In [5]:
df_scop = df_scop[df_scop["sequence"].str.len() < 1024 ]
df_scop = df_scop.sample(10000, random_state=42).reset_index(drop=True)
df_scop

,domain_id,sequence,class,fold,superfamily,family
0,d1svva_,pysfvndysvgmhpkildlmardnmtqhagygqdshcakaarlige...,c,c.67,c.67.1,c.67.1.1
1,d2f2ea1,tshkqascpvarpldvigdgwsmlivrdafegltrfgefqkslgla...,a,a.4,a.4.5,a.4.5.69
2,d2xg5a1,avsldrtravfdgseksmtldisndnkqlpylaqawienenqekii...,b,b.1,b.1.11,b.1.11.1
3,d3ddva1,nrvpssrtvsyfvakpsssemeklqlgpedsilrmerirfaddipi...,d,d.190,d.190.1,d.190.1.2
4,d1aisa1,mvdmskvklrienivasvdlfaqldlekvldlcpnskynpeefpgi...,d,d.129,d.129.1,d.129.1.1
...,...,...,...,...,...,...
9995,d1dgsa2,aeeketrlldvvfqvgrtgrvtpvgvlepvfiegsevsrvtlhnes...,b,b.40,b.40.4,b.40.4.6
9996,d1umma_,sqgkeggrdtpavkkfletgenlyiddksclrngeslfatscsgch...,a,a.3,a.3.1,a.3.1.0
9997,d1hw1a1,aqspagfaeeyiiesiwnnrfppgtilpaerelseligvtrttlre...,a,a.4,a.4.5,a.4.5.6
9998,d1exta2,scskcrkemgqveissctvdrdtvcgcrknqyrhywsenlfqcf,g,g.24,g.24.1,g.24.1.1


In [6]:
# We split into a LOOKUP set (the database) and a QUERY set (the test cases)
df_lookup, df_query = train_test_split(df_scop, test_size=0.2, random_state=SEED, shuffle=True)

print(f"Lookup Database Size: {len(df_lookup)}")
print(f"Query Set Size:       {len(df_query)}")

Lookup Database Size: 8000
Query Set Size:       2000


In [7]:
# 2. Extract Labels/Sequences Separately
# Lookup (Database)
labels_lookup = df_lookup["domain_id"].tolist()
seqs_lookup = [s.upper() for s in df_lookup["sequence"].tolist()]
families_lookup = df_lookup["family"].values
superfamilies_lookup = df_lookup["superfamily"].values
folds_lookup = df_lookup["fold"].values

# Query (Test)
labels_query = df_query["domain_id"].tolist()
seqs_query = [s.upper() for s in df_query["sequence"].tolist()]
families_query = df_query["family"].values
superfamilies_query = df_query["superfamily"].values
folds_query = df_query["fold"].values



In [8]:
# 3. Generate Embeddings Separately
print("Generating Lookup Embeddings...")
reps_lookup, strs_lookup = helper.get_hidden_representations(model, alphabet, labels_lookup, seqs_lookup)
emb_lookup = helper.get_protein_embedding(reps_lookup, strs_lookup)

print("Generating Query Embeddings...")
reps_query, strs_query = helper.get_hidden_representations(model, alphabet, labels_query, seqs_query)
emb_query = helper.get_protein_embedding(reps_query, strs_query)

# Convert to numpy for efficiency
emb_lookup = np.array(emb_lookup)
emb_query = np.array(emb_query)

Generating Lookup Embeddings...
Processing 8000 sequences in batches of 1 using multiprocessing...


100%|██████████| 8000/8000 [1:28:18<00:00,  1.51it/s]   


Function 'get_hidden_representations' executed in 4537.0609 seconds
Function 'get_protein_embedding' executed in 36.5648 seconds
Generating Query Embeddings...
Processing 2000 sequences in batches of 1 using multiprocessing...


100%|██████████| 2000/2000 [19:05<00:00,  1.75it/s]


Function 'get_hidden_representations' executed in 1145.5166 seconds
Function 'get_protein_embedding' executed in 6.5828 seconds


In [9]:
# We want to search the 20% Queries against the 100% Database
emb_full = np.concatenate([emb_lookup, emb_query], axis=0)
families_full = np.concatenate([families_lookup, families_query], axis=0)
superfamilies_full = np.concatenate([superfamilies_lookup, superfamilies_query], axis=0)
folds_full = np.concatenate([folds_lookup, folds_query], axis=0)

print(f"Full Database Size: {len(emb_full)}")

Full Database Size: 10000


### General Claim: 
If structural homology is encoded in the metric structure of the representation space, then the distance between protein embeddings reflects their degree of structural relatedness.

### Hypothesis 1:
Proteins of same superfamily are closer in the representation space then to proteins of other superfamilies.

Implication: Model understands without labels to cluster sequences into their superfamilies

In [10]:
def calculate_similarity_superfamily_split(emb_query, emb_lookup, 
                                           q_fam, q_sf, 
                                           db_fam, db_sf):
    
    print(f"Evaluating Superfamily Homology: {len(emb_query)} Queries vs {len(emb_lookup)} Database entries...")
    
    # 1. Compute Rectangular Similarity Matrix (Queries x Database)
    # Shape: (N_queries, N_lookup)
    sim_matrix = cosine_similarity(emb_query, emb_lookup)
    
    # Pre-calculate L2 distances for AUC (Shape: N_queries, N_lookup)
    l2_matrix = euclidean_distances(emb_query, emb_lookup)

    sf_auc_scores = []
    sf_hits_at_10 = []
    sf_hits_at_1 = []

    # Loop over each QUERY
    for i in range(len(emb_query)):

        # 1. POSITIVE: Lookup items in same Superfamily as Query[i]
        is_same_superfamily = db_sf == q_sf[i]

        # 2. EXCLUDED: Lookup items in same Family as Query[i]
        # (We still exclude trivial same-family matches from the database retrieval)
        is_same_family = db_fam == q_fam[i]

        # 3. VALID SEARCH SPACE: Database items NOT in the same family
        valid_search_space = ~is_same_family

        # 4. FINAL POSITIVE MASK: Same Superfamily AND Valid Search Space
        positive_mask = is_same_superfamily & valid_search_space
        
        # Check if valid positives exist in the database for this query
        if np.sum(positive_mask) == 0:
            continue
            
        # Ground Truth Labels (filtered by valid search space)
        y_true = positive_mask[valid_search_space]
        
        # Metric 1: AUC (using L2)
        # Filter scores to valid search space
        scores_l2 = l2_matrix[i][valid_search_space]
        # Invert L2 because lower distance = higher score
        try:
            auc = roc_auc_score(y_true, -scores_l2)
            sf_auc_scores.append(auc)
        except ValueError:
            # Handle edge cases where y_true is all 0 or all 1 after filtering
            continue

        # Metric 2: Hit-10 (using Cosine)
        scores_cosine = sim_matrix[i][valid_search_space]
        
        # Sort descending
        sorted_indices = np.argsort(-scores_cosine)
        
        # Check Top 10
        if np.any(y_true[sorted_indices[:10]]):
            sf_hits_at_10.append(1)
        else:
            sf_hits_at_10.append(0)

        # Check Top 1
        if np.any(y_true[sorted_indices[:1]]):
            sf_hits_at_1.append(1)
        else:
            sf_hits_at_1.append(0)

    print("\n--- SUPERFAMILY RESULTS (Strict 80/20 Split) ---")
    print(f"Mean ROC-AUC (L2):     {np.mean(sf_auc_scores):.4f}")
    print(f"Mean Hit-10 (Cosine):  {np.mean(sf_hits_at_10):.4f}")
    print(f"Mean Hit-1 (Cosine):   {np.mean(sf_hits_at_1):.4f}")

calculate_similarity_superfamily_split(
    emb_query,
    emb_full,
    families_query, superfamilies_query,
    families_full,  superfamilies_full
)

Evaluating Superfamily Homology: 2000 Queries vs 10000 Database entries...

--- SUPERFAMILY RESULTS (Strict 80/20 Split) ---
Mean ROC-AUC (L2):     0.7908
Mean Hit-10 (Cosine):  0.8565
Mean Hit-1 (Cosine):   0.7859


### Hypothesis 2:
proteins that have the same fold even if they are of different superfamilies (filter) are closer to each other in the representation space than proteins of different superfamilies which dont have the fold. 

In [11]:
def calculate_similarity_fold_split(emb_query, emb_lookup, 
                                    q_sf, q_fold, 
                                    db_sf, db_fold):
                                    
    print(f"\nEvaluating Fold Homology: {len(emb_query)} Queries vs {len(emb_lookup)} Database entries...")
    
    # Matrices
    sim_matrix = cosine_similarity(emb_query, emb_lookup)
    l2_matrix = euclidean_distances(emb_query, emb_lookup)

    fold_auc_scores = []
    fold_hits_at_10 = []

    for i in range(len(emb_query)):
        # 1. POSITIVE: Lookup items in same Fold as Query[i]
        is_same_fold = db_fold == q_fold[i]

        # 2. EXCLUDED: Lookup items in same Superfamily as Query[i]
        is_same_superfamily = db_sf == q_sf[i]

        # 3. VALID SEARCH SPACE: Database items NOT in same Superfamily
        valid_search_space = ~is_same_superfamily

        # 4. POSITIVE MASK
        positive_mask = is_same_fold & valid_search_space

        if np.sum(positive_mask) == 0:
            continue
        
        y_true = positive_mask[valid_search_space]
        
        # Metric 1: AUC (L2)
        scores_l2 = l2_matrix[i][valid_search_space]
        try:
            auc = roc_auc_score(y_true, -scores_l2)
            fold_auc_scores.append(auc)
        except ValueError:
            continue

        # Metric 2: Hit-10 (Cosine)
        scores_cosine = sim_matrix[i][valid_search_space]
        sorted_indices = np.argsort(-scores_cosine)
        
        if np.any(y_true[sorted_indices[:10]]):
            fold_hits_at_10.append(1)
        else:
            fold_hits_at_10.append(0)

    print("\n--- FOLD RESULTS (Strict 80/20 Split) ---")
    print(f"Mean ROC-AUC (L2):     {np.mean(fold_auc_scores):.4f}")
    print(f"Mean Hit-10 (Cosine):  {np.mean(fold_hits_at_10):.4f}")

# Run it
calculate_similarity_fold_split(
    emb_query,
    emb_full,
    superfamilies_query, folds_query,
    superfamilies_full,  folds_full
)


Evaluating Fold Homology: 2000 Queries vs 10000 Database entries...

--- FOLD RESULTS (Strict 80/20 Split) ---
Mean ROC-AUC (L2):     0.5808
Mean Hit-10 (Cosine):  0.5240
